# Image LSTM Prediction Trajectories

This notebook generates `image_prediction_trajectories.csv` for the final ordered Image LSTM.

Each reenactment is represented by two image sequences, one **Central** and one **Side** view. For each view sequence, the trained LSTM head is evaluated on prefixes containing the first 1, 2, ..., 30 frame features.

EfficientNetV2 features are extracted only once for the full 30-frame sequences. The prefix trajectories are then computed from these cached features, avoiding repeated image-backbone inference.

## 1. Setup

### 1.0 Add log filter

Add the log filter below or TensorFlow will print thousands of lines of uninformative log messages.

In [1]:
import re
import ipykernel.iostream

TF_LOG_FILTER_PATTERNS = [
    r'ptx\d+.*is not a recognized feature for this target',
    r'is not a recognized feature for this target \(ignoring feature\)',
    r'\(ignoring feature\)',
    r'successful NUMA node read from SysFS had negative value \(-1\)',
    r'gpu_timer\.cc:114\] Skipping the delay kernel, measurement accuracy will be reduced',
]

KERAS_PROGRESS_PATTERNS = [
    r'ms/step',
    r's/step',
    r'ETA:',
    r'\d+/\d+ \[',   # 12/64 [===>...]
]

_original_write = ipykernel.iostream.OutStream.write

def _filtered_write(self, msg, *args, **kwargs):
    text = str(msg)

    if any(re.search(p, text) for p in KERAS_PROGRESS_PATTERNS):
        _original_write(self, text, *args, **kwargs)
        return

    buf = getattr(self, '_tf_log_filter_buf', '')
    buf += text

    if '\n' not in buf:
        setattr(self, '_tf_log_filter_buf', buf)
        return

    lines = buf.splitlines(keepends=True)
    if not buf.endswith('\n'):
        incomplete = lines.pop()
    else:
        incomplete = ''

    for line in lines:
        if any(re.search(p, line) for p in TF_LOG_FILTER_PATTERNS):
            continue
        _original_write(self, line, *args, **kwargs)

    setattr(self, '_tf_log_filter_buf', incomplete)

ipykernel.iostream.OutStream.write = _filtered_write

print('Notebook log filter installed (targeted, keeps Keras steps).')

Notebook log filter installed (targeted, keeps Keras steps).


### 1.1 Imports and configuration

In [2]:
from pathlib import Path
import gc

import keras
import numpy as np
import pandas as pd
import tensorflow as tf

from keras import layers
from keras.models import Model
from keras.saving import register_keras_serializable


SEED = 13
SEQUENCE_LENGTH = 30
IMAGE_SIZE = (224, 224, 3)
BATCH_SIZE = 32

DATASET_PATH = Path('/workspace/datasets/emoji-hero-vr-db-di')
TEST_PATH = DATASET_PATH / 'test_set'
MODEL_PATH = Path('../models/image_sequence_model.keras')
OUTPUT_PATH = Path('image_prediction_trajectories.csv')

keras.utils.set_random_seed(SEED)
keras.mixed_precision.set_global_policy('mixed_float16')
tf.config.experimental.enable_op_determinism()

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

for path in [TEST_PATH, MODEL_PATH]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

2026-09-19 08:22:48.653963: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-19 08:22:48.662708: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-19 08:22:48.665603: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.17.0
Keras: 3.12.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/workspace/datasets/emoji-hero-vr-db-di/test_set: OK
../models/image_sequence_model.keras: OK


In [3]:
EMOTION_TO_ID = {
    'Anger': 0,
    'Disgust': 1,
    'Fear': 2,
    'Happiness': 3,
    'Neutral': 4,
    'Sadness': 5,
    'Surprise': 6,
}

ID_TO_EMOTION = {value: key for key, value in EMOTION_TO_ID.items()}
CLASS_NAMES = list(EMOTION_TO_ID.keys())

### 1.2 Inference-only compatibility layer

The trained model contains the custom `SequenceAugment` layer. Augmentation is disabled during inference, so this compatibility implementation only preserves the saved configuration and casts the input to `float32`.

In [4]:
@register_keras_serializable(package='seqaug')
class SequenceAugment(layers.Layer):

    def __init__(self, image_size=(224, 224), crop_scale=(0.95, 1.0), rotation_max_deg=20.0,
                 fill_mode='CONSTANT', fill_value=1.0, flip_prob=0.5, brightness_max_delta=20.0,
                 contrast_lower=0.9, contrast_upper=1.1, gamma_range=(0.9, 1.1), clip_after_color=True,
                 noise_std=4, temporal_shift_max=0, frame_drop_prob=0.05, time_mask_prob=0.05,
                 time_mask_max_frac=0.10, invert_prob=0.0, solarize_prob=0.05,
                 solarize_threshold=(120.0, 160.0), **kwargs):
        super().__init__(**kwargs)
        self.image_size = tuple(image_size)
        self.crop_scale = tuple(crop_scale)
        self.rotation_max_deg = float(rotation_max_deg)
        self.fill_mode = str(fill_mode)
        self.fill_value = float(fill_value)
        self.flip_prob = float(flip_prob)
        self.brightness_max_delta = float(brightness_max_delta)
        self.contrast_lower = float(contrast_lower)
        self.contrast_upper = float(contrast_upper)
        self.gamma_range = None if gamma_range is None else tuple(gamma_range)
        self.clip_after_color = bool(clip_after_color)
        self.noise_std = float(noise_std)
        self.temporal_shift_max = int(temporal_shift_max)
        self.frame_drop_prob = float(frame_drop_prob)
        self.time_mask_prob = float(time_mask_prob)
        self.time_mask_max_frac = float(time_mask_max_frac)
        self.invert_prob = float(invert_prob)
        self.solarize_prob = float(solarize_prob)
        self.solarize_threshold = tuple(solarize_threshold)

    def call(self, x, training=None):
        if training is True:
            raise RuntimeError('This inference-only compatibility layer must not be used for training.')
        return tf.cast(x, tf.float32)

    def get_config(self):
        config = super().get_config()
        config.update({
            'image_size': self.image_size,
            'crop_scale': self.crop_scale,
            'rotation_max_deg': self.rotation_max_deg,
            'fill_mode': self.fill_mode,
            'fill_value': self.fill_value,
            'flip_prob': self.flip_prob,
            'brightness_max_delta': self.brightness_max_delta,
            'contrast_lower': self.contrast_lower,
            'contrast_upper': self.contrast_upper,
            'gamma_range': self.gamma_range,
            'clip_after_color': self.clip_after_color,
            'noise_std': self.noise_std,
            'temporal_shift_max': self.temporal_shift_max,
            'frame_drop_prob': self.frame_drop_prob,
            'time_mask_prob': self.time_mask_prob,
            'time_mask_max_frac': self.time_mask_max_frac,
            'invert_prob': self.invert_prob,
            'solarize_prob': self.solarize_prob,
            'solarize_threshold': self.solarize_threshold,
        })
        return config


CUSTOM_OBJECTS = {
    'SequenceAugment': SequenceAugment,
    'seqaug>SequenceAugment': SequenceAugment,
}

## 2. Prepare the test image sequences

Each view is treated as its own 30-frame sequence. `sequence_id` therefore includes the camera index, while `reenactment_id` groups the corresponding Central and Side sequences.

In [5]:
def parse_sequence_dir(sequence_dir: Path) -> dict:
    parts = sequence_dir.name.split('-')

    if len(parts) != 7:
        raise ValueError(f'Unexpected sequence directory name: {sequence_dir.name}')

    sequence_timestamp, set_id, participant_id, level_id, emoji_id, emotion_id, camera_index = parts
    reenactment_id = '-'.join(parts[:-1])

    frame_paths = sorted(
        [path for path in sequence_dir.iterdir() if path.is_file()],
        key=lambda path: int(path.name.split('-')[0])
    )

    frame_timestamps = [int(path.name.split('-')[0]) for path in frame_paths]

    return {
        'sequence_id': sequence_dir.name,
        'reenactment_id': reenactment_id,
        'sequence_timestamp': int(sequence_timestamp),
        'set_id': int(set_id),
        'participant_id': int(participant_id),
        'level_id': int(level_id),
        'emoji_id': int(emoji_id),
        'emotion_id_from_sequence_id': int(emotion_id),
        'camera_index': int(camera_index),
        'perspective': 'Central' if camera_index == '0' else 'Side',
        'image_sequence_path': str(sequence_dir),
        'frame_paths': [str(path) for path in frame_paths],
        'frame_timestamps': frame_timestamps,
        'emotion_dir': sequence_dir.parent.name,
    }


sequence_rows = []

for class_dir in sorted(TEST_PATH.iterdir()):
    if not class_dir.is_dir():
        continue

    for sequence_dir in sorted(class_dir.iterdir()):
        if sequence_dir.is_dir():
            sequence_rows.append(parse_sequence_dir(sequence_dir))

sequence_df = (
    pd.DataFrame(sequence_rows)
    .sort_values(['reenactment_id', 'camera_index'])
    .reset_index(drop=True)
)

sequence_df.head()

,sequence_id,reenactment_id,sequence_timestamp,set_id,participant_id,level_id,emoji_id,emotion_id_from_sequence_id,camera_index,perspective,image_sequence_path,frame_paths,frame_timestamps,emotion_dir
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478994882, 1700478994922, 1700478994949, ...",Anger
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478994882, 1700478994922, 1700478994949, ...",Anger
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478997506, 1700478997547, 1700478997576, ...",Sadness
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700478997506, 1700478997547, 1700478997576, ...",Sadness
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,3,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...,[/workspace/datasets/emoji-hero-vr-db-di/test_...,"[1700479000163, 1700479000206, 1700479000234, ...",Happiness


In [6]:
assert len(sequence_df) == 756
assert sequence_df['sequence_id'].is_unique
assert sequence_df['reenactment_id'].nunique() == 378
assert sequence_df['participant_id'].nunique() == 8
assert set(sequence_df['camera_index']) == {0, 1}

assert sequence_df['frame_paths'].map(len).eq(SEQUENCE_LENGTH).all()
assert sequence_df['frame_timestamps'].map(len).eq(SEQUENCE_LENGTH).all()
assert (sequence_df.groupby('reenactment_id').size() == 2).all()
assert (sequence_df.groupby('reenactment_id')['camera_index'].nunique() == 2).all()
assert (sequence_df.groupby('reenactment_id')['emotion_id_from_sequence_id'].nunique() == 1).all()
assert (sequence_df['emotion_dir'].map(EMOTION_TO_ID) == sequence_df['emotion_id_from_sequence_id']).all()
assert (sequence_df.groupby('emotion_id_from_sequence_id').size() == 108).all()

X_test_paths = np.asarray(sequence_df['frame_paths'].tolist(), dtype=str)
frame_timestamps = np.asarray(sequence_df['frame_timestamps'].tolist(), dtype=np.int64)
y_test = sequence_df['emotion_id_from_sequence_id'].to_numpy(dtype=int)

assert X_test_paths.shape == (756, SEQUENCE_LENGTH)
assert frame_timestamps.shape == (756, SEQUENCE_LENGTH)
assert y_test.shape == (756,)

print('Verified 756 view sequences from 378 reenactments with 30 frames each.')

Verified 756 view sequences from 378 reenactments with 30 frames each.


In [7]:
metadata_columns = [
    'sequence_id',
    'reenactment_id',
    'sequence_timestamp',
    'set_id',
    'participant_id',
    'level_id',
    'emoji_id',
    'camera_index',
    'perspective',
    'image_sequence_path',
]

sequence_metadata = sequence_df[metadata_columns].copy()

sequence_metadata.head()

,sequence_id,reenactment_id,sequence_timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,image_sequence_path
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,/workspace/datasets/emoji-hero-vr-db-di/test_s...
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,/workspace/datasets/emoji-hero-vr-db-di/test_s...


### 2.1 Image loading

In [8]:
def parse_image(filename: tf.Tensor) -> tf.Tensor:
    image_string = tf.io.read_file(filename)
    image = tf.io.decode_jpeg(image_string, channels=IMAGE_SIZE[2])
    return image


def load_image_sequence(image_paths: tf.Tensor) -> tf.Tensor:
    images = tf.map_fn(parse_image, image_paths, fn_output_signature=tf.uint8)
    return tf.ensure_shape(images, (SEQUENCE_LENGTH, *IMAGE_SIZE))


def create_image_dataset(image_paths, batch_size=BATCH_SIZE) -> tf.data.Dataset:
    dataset = tf.data.Dataset.from_tensor_slices(image_paths)
    dataset = dataset.map(load_image_sequence, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(1)
    return dataset

## 3. Load the trained model and extract frame features once

The original model is split at `td_backbone`. EfficientNetV2 therefore processes each test frame only once. The cached frame features are then reused for all 30 prefix lengths.

In [9]:
model = keras.models.load_model(
    MODEL_PATH,
    custom_objects=CUSTOM_OBJECTS,
    compile=False,
    safe_mode=False,
)

model.trainable = False

print('Input:', model.input_shape)
print('Output:', model.output_shape)

assert model.input_shape[1:] == (SEQUENCE_LENGTH, *IMAGE_SIZE)
assert model.output_shape[-1] == len(CLASS_NAMES)

model.summary()

2026-09-19 08:23:49.433439: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Input: (None, 30, 224, 224, 3)
Output: (None, 7)


Model: "DynamicEfficientNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ video_input (InputLayer)        │ (None, 30, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequence_augment                │ (None, 30, 224, 224,   │             0 │
│ (SequenceAugment)               │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_backbone (TimeDistributed)   │ (None, 30, 1280)       │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 30, 1280)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │       168,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ top_dropout_1 (Dropout)         │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pred (Dense)                    │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,087,607 (23.22 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 6,087,607 (23.22 MB)

In [10]:
td_backbone = model.get_layer('td_backbone')

feature_and_prediction_model = Model(
    inputs=model.input,
    outputs=[td_backbone.output, model.output],
    name='image_feature_and_prediction_model',
)

frame_features, full_sequence_probabilities = feature_and_prediction_model.predict(
    create_image_dataset(X_test_paths),
    verbose=1,
)

feature_dim = int(frame_features.shape[-1])

assert frame_features.shape[:2] == (756, SEQUENCE_LENGTH)
assert full_sequence_probabilities.shape == (756, len(CLASS_NAMES))

print('frame_features:', frame_features.shape)
print('full_sequence_probabilities:', full_sequence_probabilities.shape)

24/24 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step   
frame_features: (756, 30, 1280)
full_sequence_probabilities: (756, 7)


## 4. Build the variable-length LSTM head

The prefix model reuses all trained layers after the EfficientNetV2 feature extractor. At 30 observations it must reproduce the original model predictions.

In [11]:
feature_input = layers.Input(
    shape=(None, feature_dim),
    dtype=frame_features.dtype,
    name='feature_prefix_input',
)

x = feature_input

td_backbone_index = model.layers.index(td_backbone)

for layer in model.layers[td_backbone_index + 1:]:
    x = layer(x)

prefix_model = Model(
    inputs=feature_input,
    outputs=x,
    name='image_prefix_model',
)

prefix_model.summary()

Model: "image_prefix_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ feature_prefix_input            │ (None, None, 1280)     │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, None, 1280)     │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │       168,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ top_dropout_1 (Dropout)         │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pred (Dense)                    │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168,295 (657.40 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 168,295 (657.40 KB)

In [12]:
prefix_probabilities_30 = prefix_model.predict(
    frame_features,
    batch_size=BATCH_SIZE,
    verbose=0,
)

max_difference = np.max(
    np.abs(full_sequence_probabilities - prefix_probabilities_30)
)

assert np.allclose(
    full_sequence_probabilities,
    prefix_probabilities_30,
    atol=1e-5,
)

assert np.array_equal(
    np.argmax(full_sequence_probabilities, axis=1),
    np.argmax(prefix_probabilities_30, axis=1),
)

print('Maximum prediction difference:', max_difference)
print('Full-sequence predictions verified.')

Maximum prediction difference: 0.0
Full-sequence predictions verified.


## 5. Generate prediction trajectories

For timestep `t`, the LSTM head receives only the first `t` frame-feature vectors of each view sequence. No padding or artificial observations are added.

In [13]:
n_sequences = len(frame_features)
n_classes = len(CLASS_NAMES)

prediction_trajectories = np.empty(
    (n_sequences, SEQUENCE_LENGTH, n_classes),
    dtype=np.float32,
)

for timestep in range(1, SEQUENCE_LENGTH + 1):
    feature_prefix = frame_features[:, :timestep, :]

    prediction_trajectories[:, timestep - 1, :] = prefix_model.predict(
        feature_prefix,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

print('prediction_trajectories.shape:', prediction_trajectories.shape)

prediction_trajectories.shape: (756, 30, 7)


In [14]:
assert np.allclose(
    prediction_trajectories[:, -1, :],
    full_sequence_probabilities,
    atol=1e-5,
)

assert np.allclose(
    prediction_trajectories.sum(axis=2),
    1.0,
    atol=1e-4,
)

predicted_classes = np.argmax(prediction_trajectories, axis=2)

true_class_probabilities = np.take_along_axis(
    prediction_trajectories,
    y_test[:, None, None],
    axis=2,
).squeeze(axis=2)

final_predictions = predicted_classes[:, -1]
final_correct = final_predictions == y_test

central_mask = sequence_df['camera_index'].to_numpy() == 0
side_mask = sequence_df['camera_index'].to_numpy() == 1

print(f'Final accuracy: {final_correct.sum()}/756 = {final_correct.mean():.6f}')
print(f'Central: {final_correct[central_mask].sum()}/378 = {final_correct[central_mask].mean():.6f}')
print(f'Side:    {final_correct[side_mask].sum()}/378 = {final_correct[side_mask].mean():.6f}')

assert final_correct.sum() == 550

Final accuracy: 550/756 = 0.727513
Central: 276/378 = 0.730159
Side:    274/378 = 0.724868


## 6. Export trajectory CSV

The output contains one row per view sequence and prefix length (`756 × 30 = 22,680` rows).

- `sequence_id` identifies one Central or Side 30-frame sequence.
- `reenactment_id` groups the two corresponding view sequences.
- `sample_id` uniquely identifies one trajectory point as `<sequence_id>-<timestep>`, using timesteps `01` through `30`.

In [15]:
rows = []

for sequence_index in range(n_sequences):
    metadata = sequence_metadata.iloc[sequence_index].to_dict()
    true_class_id = int(y_test[sequence_index])

    for timestep_index in range(SEQUENCE_LENGTH):
        timestep = timestep_index + 1
        predicted_class_id = int(predicted_classes[sequence_index, timestep_index])

        row = {
            'sample_id': f"{metadata['sequence_id']}-{timestep:02d}",
            **metadata,
            'timestep': timestep,
            'frame_timestamp': frame_timestamps[sequence_index, timestep_index],
            'frame_path': X_test_paths[sequence_index, timestep_index],
            'true_class_id': true_class_id,
            'true_class': ID_TO_EMOTION[true_class_id],
            'predicted_class_id': predicted_class_id,
            'predicted_class': ID_TO_EMOTION[predicted_class_id],
            'true_class_probability': true_class_probabilities[sequence_index, timestep_index],
            'predicted_class_probability': np.max(
                prediction_trajectories[sequence_index, timestep_index]
            ),
            'correct_at_timestep': predicted_class_id == true_class_id,
            'correct_final_prediction': bool(final_correct[sequence_index]),
        }

        for class_id, class_name in ID_TO_EMOTION.items():
            row[f'probability_{class_name.lower()}'] = prediction_trajectories[
                sequence_index,
                timestep_index,
                class_id,
            ]

        rows.append(row)

trajectory_df = pd.DataFrame(rows)

trajectory_df.head()

,sample_id,sequence_id,reenactment_id,sequence_timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,...,predicted_class_probability,correct_at_timestep,correct_final_prediction,probability_anger,probability_disgust,probability_fear,probability_happiness,probability_neutral,probability_sadness,probability_surprise
0,1700478995850-2-1-1-0-0-0-01,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,...,0.371928,True,True,0.371928,0.252048,0.040217,0.034443,0.028401,0.242298,0.030665
1,1700478995850-2-1-1-0-0-0-02,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,...,0.557274,True,True,0.557274,0.224796,0.010155,0.008472,0.006559,0.186028,0.006716
2,1700478995850-2-1-1-0-0-0-03,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,...,0.410943,True,True,0.410943,0.278979,0.008077,0.008118,0.006032,0.282169,0.005681
3,1700478995850-2-1-1-0-0-0-04,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,...,0.412377,True,True,0.412377,0.303184,0.007387,0.007784,0.005730,0.258307,0.005231
4,1700478995850-2-1-1-0-0-0-05,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,...,0.395946,True,True,0.395946,0.295305,0.005662,0.006280,0.004281,0.288499,0.004026


In [16]:
probability_columns = [
    f'probability_{class_name.lower()}'
    for class_name in CLASS_NAMES
]

assert len(trajectory_df) == 756 * SEQUENCE_LENGTH
assert trajectory_df['sample_id'].is_unique
assert trajectory_df['sequence_id'].nunique() == 756
assert trajectory_df['reenactment_id'].nunique() == 378

assert trajectory_df.groupby('sequence_id').size().eq(SEQUENCE_LENGTH).all()
assert trajectory_df.groupby('sequence_id')['timestep'].nunique().eq(SEQUENCE_LENGTH).all()
assert trajectory_df.groupby('reenactment_id')['sequence_id'].nunique().eq(2).all()
assert trajectory_df.groupby('reenactment_id').size().eq(2 * SEQUENCE_LENGTH).all()
assert trajectory_df.notna().all().all()

assert np.allclose(
    trajectory_df[probability_columns].sum(axis=1),
    1.0,
    atol=1e-4,
)

assert np.array_equal(
    trajectory_df['predicted_class_id'].to_numpy(),
    trajectory_df[probability_columns].to_numpy().argmax(axis=1),
)

final_rows = trajectory_df[trajectory_df['timestep'] == SEQUENCE_LENGTH]

assert len(final_rows) == 756
assert final_rows['correct_at_timestep'].sum() == 550
assert final_rows['sequence_id'].is_unique

print('Trajectory table passed all integrity checks.')

Trajectory table passed all integrity checks.


In [17]:
trajectory_df.to_csv(OUTPUT_PATH, index=False)

print(f'Exported {len(trajectory_df):,} rows to {OUTPUT_PATH.resolve()}')
print(f'Columns: {len(trajectory_df.columns)}')

Exported 22,680 rows to /workspace/repos/emohevrdb-dfer/5_dynamic_facial_expression_recognition/5_1_image_sequence_based_fer/5_1_4_sequence_order_ablation/trajectories/image_prediction_trajectories.csv
Columns: 29
